# 01 — Data quality and train journey reconstruction

**Goal :** Before building labels, we check how trustworthy the actual data is, and how to rebuild each train's journey.

1. Dataset overview
2. Defining the scope: IC / IR / RE / EC
3. Swiss vs foreign stops
4. Measured times (`REAL`) by category, operator, station and day
5. Cancellations and extra trips
6. Delay sanity checks
7. Journey reconstruction
8. Prediction points preview
9. Findings and decisions


In [1]:
# 0. Setup: imports, chart style, DuckDB view `trains` and helper q()
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

from swissdelay import config

pd.set_option("display.max_rows", 60)
pd.set_option("display.float_format", "{:,.3f}".format)

# Chart style
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
GRAY, LIGHT_GRAY = "#8a8984", "#d4d3cd"
pio.templates["swissdelay"] = go.layout.Template(
    layout=dict(
        colorway=SERIES,
        font=dict(family="Inter, system-ui, sans-serif", size=13, color="#0b0b0b"),
        paper_bgcolor="#fcfcfb",
        plot_bgcolor="#fcfcfb",
        xaxis=dict(gridcolor="#e8e7e2", zeroline=False, linecolor="#c3c2b7"),
        yaxis=dict(gridcolor="#e8e7e2", zeroline=False, linecolor="#c3c2b7"),
        bargap=0.25,
        margin=dict(l=60, r=30, t=60, b=50),
    )
)
pio.templates.default = "plotly_white+swissdelay"

SCOPE_CATEGORIES = ("IC", "IR", "RE", "EC")
CAT_COLORS = dict(zip(SCOPE_CATEGORIES, SERIES))
STATUS_COLORS = {"REAL": SERIES[0],"PROGNOSE": SERIES[1], "GESCHAETZT": SERIES[3],
                 "UNBEKANNT": GRAY, "(empty)": LIGHT_GRAY}

con = duckdb.connect()
con.execute(f"""
CREATE OR REPLACE VIEW trains AS
SELECT
  *,
  operating_day::VARCHAR || '|' || trip_id                 AS trip_key,
  station_id // 100000 = 85                                AS is_swiss,
  upper(regexp_extract(line_name, '^[A-Za-z]+'))           AS line_prefix,
  date_diff('second', arr_planned, arr_actual) / 60.0      AS arr_delay_min,
  date_diff('second', dep_planned, dep_actual) / 60.0      AS dep_delay_min
FROM read_parquet('{config.INTERIM}/trains_*.parquet')
""")
SCOPE = f"category IN {SCOPE_CATEGORIES}"


def q(sql: str) -> pd.DataFrame:
    """Run SQL in DuckDB and return a pandas DataFrame."""
    return con.sql(sql).df()

## 1. Dataset overview

In [2]:
# 1.1 Rows, days, train runs and stations per month
overview = q("""
SELECT strftime(operating_day, '%Y-%m') AS month,
       count(DISTINCT operating_day)    AS days,
       count(*)                         AS rows,
       count(DISTINCT trip_key)         AS trips,
       count(DISTINCT station_id)       AS stations
FROM trains GROUP BY ALL ORDER BY month
""")
overview

,month,days,rows,trips,stations
0,2025-08,31,5528252,491961,2362
1,2025-09,30,5400412,475348,2362
2,2025-10,31,5478446,475898,2344
3,2025-11,30,5334974,458800,2337
4,2025-12,31,5530928,475180,2376
5,2026-01,31,5570335,484098,2338
6,2026-02,28,5080328,444274,2324
7,2026-03,31,5559780,487669,2328
8,2026-04,30,5329820,468966,2332
9,2026-05,31,5528881,484176,2322


> **Finding 1.1.**
- 426 days (1 Aug 2025 – 30 Sep 2026), none missing.
- Each month has 5.1–5.6 M train rows and 444–492 k train runs (≈ 15–16 k runs per day, all train types).
- About 2,280–2,380 stations appear each month, Swiss and foreign; the count drops by ≈ 50 from July 2026, probably a timetable or network change rather than missing data.

## 2. Defining the scope: IC / IR / RE / EC

`category` is the official train category

In [3]:
# 2.1 Rows and train runs per category (all trains)
categories = q("""
SELECT category, count(*) AS rows, count(DISTINCT trip_key) AS trips,
       round(100 * count(*) / sum(count(*)) OVER (), 2) AS pct_rows
FROM trains GROUP BY ALL ORDER BY rows DESC
""")
categories.head(25)

,category,rows,trips,pct_rows
0,S,46481247,3679118,61.140
1,R,15296333,1423350,20.120
2,RE,5007749,561386,6.590
3,IR,3541391,386984,4.660
4,IC,1440014,179002,1.890
5,TER,1351534,158077,1.780
6,RB,892401,70341,1.170
7,PE,547830,44117,0.720
8,ICE,349858,24420,0.460
9,EC,301205,30212,0.400


In [4]:
# 2.2 Chart: train runs per in-scope category
fig = px.bar(
    categories[categories["category"].isin(SCOPE_CATEGORIES)],
    x="category", y="trips", color="category", color_discrete_map=CAT_COLORS,
    title="Train runs in scope, Aug 2025 – Sep 2026", text_auto=".3s",
)
fig.update_layout(showlegend=False, yaxis_title="train runs", xaxis_title=None)
fig.show()

> **Finding 2.1–2.2.**
- S-Bahn (61 % of rows) and regional R trains (20 %) dominate.
- The scope IC / IR / RE / EC is 10.3 M rows (13.5 %) and **1.16 M train runs** (≈ 2,700 per day): RE 561 k, IR 387 k, IC 179 k, EC 30 k.
- International long-distance trains (ICE, TGV, RJX, RJ, NJ, EN; ≈ 67 k runs) are **network context only**, not prediction targets. 131 rows have no category (one-stop junk).

In [5]:
# 2.3 Category vs line-name prefix: are in-scope trains hidden under another category?
crosstab = q(f"""
SELECT category, line_prefix, count(*) AS rows, count(DISTINCT trip_key) AS trips
FROM trains
WHERE category IN {SCOPE_CATEGORIES} OR line_prefix IN {SCOPE_CATEGORIES}
GROUP BY ALL ORDER BY rows DESC
""")

# Rows where category and line prefix disagree are the ones to look at
crosstab[crosstab["category"] != crosstab["line_prefix"]].head(30)

,category,line_prefix,rows,trips
4,IR,IRVAE,213435,19469
5,IR,IRLEX,114397,15975
6,RE,RERE,66103,4470
7,RE,REN,8942,1239
8,IR,IRLIX,8128,1205
9,Zug,RE,1479,154
10,IR,IRN,902,180
11,RE,RERB,721,46
12,Zug,IC,486,30
13,R,RE,50,6


> **Finding 2.3.**
- `category` alone defines the scope.
- Disagreements with the line-name prefix are either named services already in scope (IR VAE = Voralpen-Express, IR LEX / LIX = Luzern–Engelberg / Luzern–Interlaken Express, night variants), or ≈ 190 runs with category `Zug` / `R` and an RE / IC line name (0.02 %, mostly German), which we ignore.
- For a "line" variable later, use the full `line_name`, not the prefix.

## 3. Swiss vs foreign stops

Swiss stations have a `station_id` starting with 85. A trip is *domestic* (all stops in Switzerland), *cross-border*, or *foreign only* (e.g. the DB RE7 around Augsburg).

In [6]:
# 3.1 Domestic / cross-border / foreign-only runs per category (% of runs)
geo = q(f"""
WITH t AS (
  SELECT trip_key, any_value(category) AS category,
         bool_and(is_swiss) AS all_ch, bool_or(is_swiss) AS any_ch
  FROM trains WHERE {SCOPE} GROUP BY trip_key
)
SELECT category,
       CASE WHEN all_ch THEN 'domestic' WHEN any_ch THEN 'cross-border' ELSE 'foreign only' END AS kind,
       count(*) AS trips
FROM t GROUP BY ALL ORDER BY category, kind
""")
geo_wide = geo.pivot(index="category", columns="kind", values="trips").fillna(0)
geo_wide.div(geo_wide.sum(axis=1), axis=0).mul(100).round(1)

kind,cross-border,domestic,foreign only
category,,,
EC,92.700,6.300,0.900
IC,7.800,90.200,1.900
IR,4.500,95.500,0.000
RE,18.200,73.800,8.000


In [7]:
# 3.2 Chart: domestic / cross-border / foreign-only runs per category
fig = px.bar(
    geo, x="category", y="trips", color="kind", barmode="stack",
    color_discrete_map={"domestic": SERIES[0], "cross-border": SERIES[1], "foreign only": GRAY},
    category_orders={"kind": ["domestic", "cross-border", "foreign only"]},
    title="Where in-scope trains run",
)
fig.update_layout(yaxis_title="train runs", xaxis_title=None, legend_title=None)
fig.show()

> **Finding 3.1–3.2.**
- IR is 95.5 % domestic, IC 90.2 %, RE 73.8 %; EC is 92.7 % cross-border. ≈ 49 k runs never stop in Switzerland (RE 8.0 %, IC 1.9 %, EC 0.9 %).

In [8]:
# 3.3 Operators of foreign-only runs (excluded from targets)
q(f"""
WITH t AS (SELECT trip_key, any_value(operator_abbr) AS operator, any_value(category) AS category,
                  bool_or(is_swiss) AS any_ch
           FROM trains WHERE {SCOPE} GROUP BY trip_key)
SELECT operator, category, count(*) AS trips FROM t WHERE NOT any_ch
GROUP BY ALL ORDER BY trips DESC LIMIT 15
""")

,operator,category,trips
0,DB,RE,32415
1,ÖBB,RE,9616
2,DB,IC,3183
3,DB Regio,RE,1754
4,SBB,RE,864
5,ÖBB,IC,218
6,MBS,RE,182
7,ÖBB,EC,144
8,ZSSK,EC,109
9,Trenitalia,EC,12


> **Finding 3.3.**
- Foreign-only runs are German and Austrian regional trains (DB, DB Regio, ÖBB, MBS), German ICs, foreign sections of ECs (ÖBB, ZSSK, ČD, Trenitalia), and ≈ 870 SBB runs that stay in Germany.
- All are dropped.

In [9]:
# 3.4 Share of REAL departures at Swiss vs foreign stops
q(f"""
SELECT is_swiss, count(*) AS dep_events,
       avg((dep_status = 'REAL')::INT) AS share_real
FROM trains WHERE {SCOPE} AND dep_planned IS NOT NULL
GROUP BY ALL
""")

,is_swiss,dep_events,share_real
0,True,7783836,0.957
1,False,1348025,0.007


> **Finding 3.4.**
- Measured times exist only at Swiss stops: **95.7 % `REAL` vs 0.7 % abroad**.
- Foreign stops carry no observed delay, so journeys are restricted to Swiss stops: cross-border runs keep their Swiss section, flagged `enters_from_abroad`, and foreign-only runs disappear.
- Caveat: `is_swiss` uses the station-number prefix 85, which also covers a few stations in Italy (see 4.6).

## 4. Measured times (`REAL`)

Each actual time comes with a status:

| Status | Meaning | Counts as observed? |
|---|---|---|
| `REAL` | Measured: the train was actually detected leaving or arriving, by track sensors or the train's own reporting | ✅ Yes, the only one |
| `PROGNOSE` | Forecast: the last time the system predicted for that stop, never confirmed by a measurement | ❌ No |
| `UNBEKANNT` | Unknown: the system has neither a measurement nor a forecast | ❌ No |
| `GESCHAETZT` | Estimated (does not appear in our scope) | ❌ No |

**How we use it.** 
- We keep **every row** of a journey, because the planned times give the stop order, the target stop of each horizon and the timetable slack.
- But only `REAL` times count as observed: a delay computed from a `PROGNOSE` or `UNBEKANNT` time is treated as missing.
- Prediction points are `REAL` departures, and a label needs a `REAL` time at the target stop.
- Dropping the non-`REAL` rows instead would remove stops from the sequence and change which stop is the target, which would bias the labels.

Shares below are computed on stops that **have** a planned time (the first stop has no arrival, the last no departure), Swiss stops only.

In [10]:
# 4.1 Status of actual times per category, arrivals and departures (% of stops)
status = q(f"""
SELECT category, 'departure' AS event, coalesce(dep_status, '(empty)') AS status, count(*) AS n
FROM trains WHERE {SCOPE} AND is_swiss AND dep_planned IS NOT NULL GROUP BY ALL
UNION ALL
SELECT category, 'arrival', coalesce(arr_status, '(empty)'), count(*)
FROM trains WHERE {SCOPE} AND is_swiss AND arr_planned IS NOT NULL GROUP BY ALL
""")
status["share"] = status["n"] / status.groupby(["category", "event"])["n"].transform("sum")
status.pivot_table(index=["category", "event"], columns="status", values="share").mul(100).round(1)

status              PROGNOSE   REAL  UNBEKANNT
category event                                
EC       arrival       2.400 90.000      7.600
         departure     2.400 90.200      7.300
IC       arrival       0.500 94.600      4.900
         departure     0.700 94.700      4.600
IR       arrival       0.300 96.100      3.600
         departure     0.400 96.100      3.500
RE       arrival       1.800 95.800      2.400
         departure     1.800 95.800      2.400

In [11]:
# 4.2 Chart: status of actual times per category
fig = px.bar(
    status, x="share", y="category", color="status", facet_col="event", orientation="h",
    color_discrete_map=STATUS_COLORS,
    category_orders={"status": list(STATUS_COLORS), "category": list(SCOPE_CATEGORIES)},
    title="Status of actual times, Swiss stops", hover_data={"n": ":,"},
)
fig.update_xaxes(tickformat=".0%", title=None)
fig.update_layout(legend_title=None, yaxis_title=None)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

> **Finding 4.1–4.2.**
- At Swiss stops, 94.6–96.1 % of IC, IR and RE times are `REAL`, EC 90 %.
- The rest is mostly `UNBEKANNT` (2.4–7.6 %); `PROGNOSE` is at most 2.4 % and `GESCHAETZT` does not occur.
- Arrivals and departures behave the same.

In [12]:
# 4.3 Share of REAL departures per operator
by_operator = q(f"""
SELECT operator_abbr, any_value(operator_name) AS operator_name,
       count(*) AS dep_events,
       avg((dep_status = 'REAL')::INT) AS share_real
FROM trains WHERE {SCOPE} AND is_swiss AND dep_planned IS NOT NULL
GROUP BY operator_abbr ORDER BY dep_events DESC
""")
by_operator.head(20)

,operator_abbr,operator_name,dep_events,share_real
0,SBB,Schweizerische Bundesbahnen SBB,4592179,0.954
1,SOB-sob,Schweizerische Südostbahn (sob),818149,0.961
2,RhB,Rhätische Bahn,816979,0.991
3,BLS-bls,BLS AG (bls),724002,0.953
4,RBS,Regionalverkehr Bern-Solothurn,269196,0.987
5,TPF,Transports publics fribourgeois,213434,0.920
6,ZB,Zentralbahn,105394,0.947
7,THURBO,THURBO,96564,0.985
8,MGB-bvz,Matterhorn Gotthard Bahn (bvz),83011,0.982
9,FART,Ferrovie Autolinee Regionali Ticinesi,40634,0.603


> **Finding 4.3.**
- All large operators are between 92 % and 99 % `REAL` (SBB 95.4 %, SOB 96.1 %, RhB 99.1 %, BLS 95.3 %, RBS 98.7 %, TPF 92.0 %).
- Three are poorly measured: FART 60 % (Centovalli line, 40.6 k departures), DB Regio 78 % (18.3 k) and DB 0 % (5.9 k).
- **Decision:** we exclude FART, DB and DB Regio from prediction points and targets (0.8 % of Swiss departures); we keep them as network context.

In [13]:
# 4.4 Share of REAL departures per station
by_station = q(f"""
SELECT station_id, any_value(station_name) AS station_name,
       count(*) AS dep_events,
       avg((dep_status = 'REAL')::INT) AS share_real
FROM trains WHERE {SCOPE} AND is_swiss AND dep_planned IS NOT NULL
GROUP BY station_id ORDER BY dep_events DESC
""")
print(f"{len(by_station)} Swiss stations served by IC/IR/RE/EC")
print("Stations with < 50 % REAL departures:", (by_station["share_real"] < 0.5).sum())
by_station.head(20)

729 Swiss stations served by IC/IR/RE/EC
Stations with < 50 % REAL departures: 75


,station_id,station_name,dep_events,share_real
0,8503000,Zürich HB,209332,0.956
1,8507000,Bern,167402,0.958
2,8500218,Olten,152181,0.963
3,8501120,Lausanne,116684,0.927
4,8501118,Renens VD,109575,0.935
5,8501008,Genève,102682,0.904
6,8503016,Zürich Flughafen,92603,0.914
7,8506000,Winterthur,89663,0.967
8,8505000,Luzern,79306,0.975
9,8506302,St. Gallen,77890,0.982


> **Finding 4.4.** 
- 729 stations with a Swiss number are served by the scope.
- The busiest are all ≥ 90 % `REAL` (Zürich HB 95.6 %, Bern 95.8 %, Genève 90.4 %, St. Gallen 98.2 %).
- 75 stations are below 50 %, all small.

In [14]:
# 4.5 Chart: distribution of the REAL share across stations
bins = q(f"""
WITH s AS (
  SELECT station_id, avg((dep_status = 'REAL')::INT) AS share_real
  FROM trains WHERE {SCOPE} AND is_swiss AND dep_planned IS NOT NULL
  GROUP BY station_id HAVING count(*) >= 1000
)
SELECT floor(share_real * 20) / 20 AS bin, count(*) AS stations FROM s GROUP BY ALL ORDER BY bin
""")
fig = px.bar(bins, x="bin", y="stations", title="Share of REAL departures per station (≥ 1,000 departures)")
fig.update_traces(marker_color=SERIES[0])
fig.update_layout(xaxis_title="share REAL (5 % bins)", yaxis_title="stations", xaxis_tickformat=".0%")
fig.show()

> **Finding 4.5.**
- Of the 357 stations with ≥ 1,000 departures, 280 are ≥ 95 % `REAL`, 48 between 90 and 95 %, and 29 below 90 % (4 at 0 %).

In [15]:
# 4.6 Poorly measured stations (≥ 1,000 departures, < 80 % REAL)
q(f"""
SELECT station_name, mode(operator_abbr) AS main_operator, count(*) AS dep_events,
       avg((dep_status = 'REAL')::INT) AS share_real
FROM trains WHERE {SCOPE} AND is_swiss AND dep_planned IS NOT NULL
GROUP BY station_name HAVING count(*) >= 1000 AND share_real < 0.8
ORDER BY dep_events DESC, station_name
""")

,station_name,main_operator,dep_events,share_real
0,Basel Bad Bf,SBB,15996,0.000
1,Iselle di Trasquera,BLS-bls,9438,0.537
2,Varzo,BLS-bls,8415,0.000
3,Preglia,BLS-bls,8412,0.000
4,Masera,FART,2765,0.599
5,Trontano,FART,2765,0.599
6,Camedo,FART,2763,0.571
7,Intragna,FART,2763,0.605
8,Druogno,FART,2758,0.604
9,Malesco,FART,2758,0.605


> **Finding 4.6.**
- The poorly measured stations are of three kinds: (1) the FART Centovalli / Vigezzina line (Intragna, Camedo, Re, Malesco… 45–70 %), removed by the operator exclusion; (2) stations **in Italy** that carry a Swiss number: Varzo and Preglia (0 %) and Iselle (54 %) on the Simplon line, Tirano (0 %) on the Bernina line; (3) Basel Bad Bf (0 %), the German-run station in Basel. Ebikon (75 %) and Stansstad (76 %) are ordinary Swiss stations with gaps.
- So `is_swiss` is not exactly "measured".
- **Decision :** a station is a valid prediction / target station if ≥ 80 % of its in-scope departures are `REAL` over the training months (Aug 2025 – Apr 2026); other stations stay in journeys as context.

In [16]:
# 4.7 Daily share of REAL departures and possible feed outages
daily_real = q(f"""
SELECT operating_day, category, avg((dep_status = 'REAL')::INT) AS share_real, count(*) AS dep_events
FROM trains WHERE {SCOPE} AND is_swiss AND dep_planned IS NOT NULL
GROUP BY ALL ORDER BY operating_day
""")
fig = px.line(daily_real, x="operating_day", y="share_real", color="category",
              color_discrete_map=CAT_COLORS, title="Daily share of REAL departures")
fig.update_layout(yaxis_tickformat=".0%", xaxis_title=None, yaxis_title=None, legend_title=None,
                  hovermode="x unified")
fig.show()

# Days where measurement dropped (possible feed outages)
d = daily_real.groupby("operating_day").apply(
    lambda g: (g["share_real"] * g["dep_events"]).sum() / g["dep_events"].sum()
).rename("share_real").reset_index()
thr = d["share_real"].mean() - 3 * d["share_real"].std()
d[d["share_real"] < thr]

,operating_day,share_real
43,2025-09-13,0.867
44,2025-09-14,0.869
199,2026-02-16,0.870
316,2026-06-13,0.875
317,2026-06-14,0.865
414,2026-09-19,0.876
421,2026-09-26,0.853
422,2026-09-27,0.864


> **Finding 4.7.**
- No feed outage: the daily share never drops below 85 % (mean IC 95 %, IR 96 %, RE 96 %, EC 91 %).
- The 8 days below mean − 3 sd (85–88 %) are weekends except Monday 16 Feb 2026, likely planned engineering works; to cross-check with cancellations in section 5.
- EC is noisy (≈ 70 runs per day; low of 63 % on 12 Jan 2026).

## 5. Cancellations and extra trips

A trip is *fully cancelled* if every stop is cancelled, *partially cancelled* if only some are (e.g. a train turned back early). Days with many cancellations are candidates for the **disruption days** breakdown.

In [17]:
# 5.1 Fully / partially cancelled and extra runs per day (% of in-scope runs)
cancel_daily = q(f"""
WITH t AS (
  SELECT operating_day, trip_key, bool_and(is_cancelled) AS full_c, bool_or(is_cancelled) AS any_c,
         bool_or(is_extra_trip) AS extra
  FROM trains WHERE {SCOPE} GROUP BY ALL
)
SELECT operating_day, count(*) AS trips,
       avg(full_c::INT) AS fully_cancelled,
       avg((any_c AND NOT full_c)::INT) AS partially_cancelled,
       avg(extra::INT) AS extra_trips
FROM t GROUP BY ALL ORDER BY operating_day
""")
# Distribution over the 426 days, in % of in-scope runs (count row dropped: it is not a share)
cancel_daily[["fully_cancelled", "partially_cancelled", "extra_trips"]].describe().drop("count").mul(100).round(2)

,fully_cancelled,partially_cancelled,extra_trips
mean,0.990,6.090,2.940
std,0.850,3.900,2.350
min,0.080,0.820,0.300
25%,0.450,2.700,1.280
50%,0.730,5.100,2.200
75%,1.280,9.020,3.690
max,5.500,18.240,12.410


> **Finding 5.1.** Over 426 days, on average **1.0 %** of in-scope runs are fully cancelled (median 0.7 %, max 5.5 %), **6.1 %** are partially cancelled, i.e. at least one stop cancelled (median 5.1 %, max 18.2 %), and **2.9 %** are extra runs (max 12.4 %). Caveat: this section counts all stops, including foreign ones, so partial cancellations on the Swiss section alone will be a bit lower.

In [18]:
# 5.2 Chart: daily share of cancelled runs + the 10 days with most fully cancelled runs
long = cancel_daily.melt(id_vars="operating_day",
                         value_vars=["fully_cancelled", "partially_cancelled"],
                         var_name="type", value_name="share")
fig = px.line(long, x="operating_day", y="share", color="type",
              color_discrete_map={"fully_cancelled": SERIES[7], "partially_cancelled": SERIES[1]},
              title="Share of in-scope train runs cancelled, per day")
fig.update_layout(yaxis_tickformat=".1%", xaxis_title=None, yaxis_title=None, legend_title=None,
                  hovermode="x unified")
fig.show()
cancel_daily.nlargest(10, "fully_cancelled")

,operating_day,trips,fully_cancelled,partially_cancelled,extra_trips
274,2026-05-02,2820,0.055,0.118,0.095
275,2026-05-03,2767,0.052,0.122,0.094
421,2026-09-26,2878,0.049,0.123,0.110
422,2026-09-27,2826,0.048,0.107,0.108
366,2026-08-02,2798,0.045,0.116,0.076
254,2026-04-12,2838,0.044,0.098,0.096
365,2026-08-01,2822,0.043,0.104,0.072
380,2026-08-16,2802,0.037,0.140,0.080
100,2025-11-09,2807,0.036,0.097,0.092
379,2026-08-15,2816,0.036,0.125,0.072


In [19]:
# 5.3 Cancellations by day of week and by month (% of in-scope runs)
cd = cancel_daily.assign(
    weekday=pd.to_datetime(cancel_daily["operating_day"]).dt.day_name(),
    month=pd.to_datetime(cancel_daily["operating_day"]).dt.strftime("%Y-%m"),
)
cols = ["fully_cancelled", "partially_cancelled", "extra_trips"]
days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
display(cd.groupby("weekday")[cols].mean().reindex(days).mul(100).round(2))
cd.groupby("month")[cols].mean().mul(100).round(2)

,fully_cancelled,partially_cancelled,extra_trips
weekday,,,
Monday,0.780,4.840,2.230
Tuesday,0.740,4.650,2.010
Wednesday,0.720,4.590,1.970
Thursday,0.780,4.930,2.170
Friday,0.860,4.730,2.240
Saturday,1.540,9.400,4.930
Sunday,1.530,9.460,5.030


,fully_cancelled,partially_cancelled,extra_trips
month,,,
2025-08,0.670,8.850,3.840
2025-09,0.970,4.830,3.590
2025-10,0.670,5.240,3.460
2025-11,0.930,5.660,2.990
2025-12,0.550,2.440,1.330
2026-01,0.820,2.290,1.930
2026-02,0.810,7.250,2.080
2026-03,0.840,5.990,2.120
2026-04,0.720,4.920,2.620


In [20]:
# 5.4 Days with the most partially cancelled runs (compare with the low-REAL days of 4.7)
cancel_daily.assign(weekday=pd.to_datetime(cancel_daily["operating_day"]).dt.day_name()) \
    .nlargest(10, "partially_cancelled")

,operating_day,trips,fully_cancelled,partially_cancelled,extra_trips,weekday
93,2025-11-02,2889,0.017,0.182,0.123,Sunday
317,2026-06-14,2748,0.035,0.173,0.068,Sunday
199,2026-02-16,2848,0.020,0.172,0.069,Monday
92,2025-11-01,2920,0.022,0.171,0.117,Saturday
200,2026-02-17,2870,0.021,0.170,0.079,Tuesday
86,2025-10-26,2925,0.009,0.160,0.124,Sunday
393,2026-08-29,2705,0.011,0.151,0.037,Saturday
37,2025-09-07,2854,0.015,0.149,0.106,Sunday
316,2026-06-13,2761,0.028,0.147,0.057,Saturday
232,2026-03-21,2775,0.012,0.146,0.071,Saturday


> **Finding 5.2–5.4.** Cancellations are a **weekend phenomenon**: fully cancelled runs double on Saturdays and Sundays (≈ 1.5 % vs 0.75 % on weekdays), partially cancelled runs too (≈ 9.4 % vs 4.7 %). The 10 worst days for full cancellations are all weekends (2–3 May, 1–2 Aug, 15–16 Aug, 26–27 Sep 2026, 12 Apr 2026, 9 Nov 2025), and they also have many extra runs (7–11 %), i.e. replacement services. Partial cancellations peak in summer (June–August ≈ 8.3–9.0 %) and are lowest in December–January (≈ 2.3–2.4 %), the construction-season pattern. The days with the most partial cancellations (1–2 Nov 2025, 13–14 Jun 2026, 16–17 Feb 2026) are **the same days as the `REAL` dips in 4.7**: those dips are **planned engineering works** (short-turned or diverted trains), not feed outages; 16–17 Feb 2026 (Mon–Tue) is probably an incident. 26 Oct 2025 (16 % partial, 12.4 % extra runs, the maximum) is also the night clocks went back: checked in 7.8: not a daylight-saving artefact (same pattern on Saturday 25 Oct), a weekend of works.
>
> Full cancellations are higher from May 2026 (1.3–1.6 % per month vs 0.6–0.9 % before): **the validation (May) and test (June) months are more disrupted than the training months**, a realistic stress test that will make errors there look somewhat worse.
>
> **For the disruption-day breakdown:** cancellations mostly flag *planned* works. Following the roadmap, disruption days = worst 5 % of days (≈ 21 days) by a combination of mean delay and cancellations, to be fixed after section 6.

## 6. Delay sanity checks

Departure delay, REAL only, Swiss stops. Planned times are rounded to the minute, so small values (< 30 s) are partly rounding noise.

In [21]:
# 6.1 Departure delay statistics per category (REAL, Swiss stops, minutes)
q(f"""
SELECT category,
       count(*) AS n,
       avg(dep_delay_min) AS mean,
       min(dep_delay_min) AS min,
       quantile_cont(dep_delay_min, 0.01) AS p01,
       quantile_cont(dep_delay_min, 0.10) AS p10,
       quantile_cont(dep_delay_min, 0.50) AS p50,
       quantile_cont(dep_delay_min, 0.90) AS p90,
       quantile_cont(dep_delay_min, 0.99) AS p99,
       max(dep_delay_min) AS max,
       avg((dep_delay_min < -1)::INT)  AS share_early_1min,
       avg((dep_delay_min > 3)::INT)   AS share_over_3min,
       avg((dep_delay_min > 60)::INT)  AS share_over_60min
FROM trains WHERE {SCOPE} AND is_swiss AND dep_status = 'REAL'
GROUP BY ALL ORDER BY category
""")

,category,n,mean,min,p01,p10,p50,p90,p99,max,share_early_1min,share_over_3min,share_over_60min
0,EC,131554,4.910,"-1,437.317",-0.067,0.450,1.483,11.617,52.949,313.867,0.002,0.293,0.006
1,IC,1077424,1.554,-27.867,0.200,0.417,0.933,3.033,10.967,151.000,0.001,0.102,0.000
2,IR,3017542,1.268,-45.367,0.000,0.217,0.750,2.667,8.017,190.000,0.001,0.079,0.000
3,RE,3218984,-19.209,"-66,110,897.000",-0.033,0.183,0.783,2.833,8.900,320.683,0.002,0.089,0.000


In [22]:
# 6.2 Chart: departure delay distribution per category (clipped to [-3, 20] min)
hist = q(f"""
SELECT category, greatest(least(floor(dep_delay_min), 20), -3) AS minute, count(*) AS n
FROM trains WHERE {SCOPE} AND is_swiss AND dep_status = 'REAL'
GROUP BY ALL ORDER BY minute
""")
hist["share"] = hist["n"] / hist.groupby("category")["n"].transform("sum")
fig = px.bar(hist, x="minute", y="share", color="category", barmode="group",
             color_discrete_map=CAT_COLORS, category_orders={"category": list(SCOPE_CATEGORIES)},
             title="Departure delay distribution (clipped to [-3, 20] min)")
fig.update_layout(yaxis_tickformat=".0%", xaxis_title="delay (min, floored)", yaxis_title=None,
                  legend_title=None)
fig.show()

> **Finding 6.1–6.2.** For normal rows the delay distribution is sound. Median departure delay is under 1 min for IC, IR and RE (0.75–0.93 min), 1.5 min for EC; 53–60 % of IC / IR / RE departures are less than 1 min late, ≈ 37 % for EC. 90 % of departures are less than 2.7–3.0 min late (IC / IR / RE) and 11.6 min (EC); 99 % less than 8–11 min (IC / IR / RE) and 53 min (EC). Departures more than 3 min late: IC 10 %, IR 8 %, RE 9 %, **EC 29 %** (6 % of EC departures are ≥ 20 min late: delay imported from abroad). Early departures (> 1 min) are rare (0.1–0.2 %). **The RE mean of −19.2 min comes from a single corrupted row** (−66 million min, see 6.4); without it the RE mean is ≈ +1.3 min. The minima (EC −1,437, IR −45, IC −28 min) are also artefacts, examined in 6.4–6.5.

In [23]:
# 6.3 Impossible or suspicious combinations (all in-scope stops, row counts)
q(f"""
SELECT
  count(*) FILTER (WHERE arr_status = 'REAL' AND dep_status = 'REAL' AND dep_actual < arr_actual)
    AS dep_before_arr,
  count(*) FILTER (WHERE arr_planned > dep_planned)                AS planned_dep_before_arr,
  count(*) FILTER (WHERE dep_status = 'REAL' AND dep_delay_min < -5) AS early_over_5min,
  count(*) FILTER (WHERE dep_status = 'REAL' AND dep_delay_min > 180) AS late_over_3h,
  count(*) FILTER (WHERE is_pass_through)                           AS pass_through_rows,
  count(*) FILTER (WHERE is_pass_through AND dep_status = 'REAL')   AS pass_through_with_real
FROM trains WHERE {SCOPE}
""").T

,0
dep_before_arr,39201
planned_dep_before_arr,3961
early_over_5min,151
late_over_3h,31
pass_through_rows,11586
pass_through_with_real,9843


> **Finding 6.3.** Suspicious rows, all in-scope stops: 151 REAL departures more than 5 min early, 31 more than 3 h late, 39,201 departures measured before the arrival at the same stop, 3,961 rows whose planned departure is before the planned arrival (timetable inconsistency, to check around midnight in section 7), and 11,586 pass-through rows (the train passes without stopping), 9,843 of them with a REAL departure.

In [24]:
# 6.4 The most negative REAL departure delays: what is wrong with them?
q(f"""
SELECT operating_day, category, operator_abbr, line_name, station_name,
       dep_planned, dep_actual, round(dep_delay_min, 1) AS dep_delay_min,
       arr_planned, arr_actual, arr_status
FROM trains
WHERE {SCOPE} AND dep_status = 'REAL' AND dep_delay_min < -5
ORDER BY dep_delay_min
LIMIT 20
""")

,operating_day,category,operator_abbr,line_name,station_name,dep_planned,dep_actual,dep_delay_min,arr_planned,arr_actual,arr_status
0,2025-09-10,RE,RBS,RE5,Lohn-Lüterkofen,2025-09-10 08:17:00,1899-12-30 00:00:00,"-66,110,897.000",2025-09-10 08:17:00,2025-09-10 08:16:38,REAL
1,2026-01-08,EC,SBB,EC,Basel SBB,2026-01-09 07:38:00,2026-01-08 07:40:41,"-1,437.300",2026-01-09 07:21:00,2026-01-08 07:20:57,REAL
2,2026-02-21,EC,SBB,EC,Basel SBB,2026-02-22 07:38:00,2026-02-21 09:14:51,"-1,343.200",2026-02-22 07:21:00,2026-02-21 08:52:27,REAL
3,2026-06-16,RE,RhB,RE38,Thusis,2026-06-16 06:28:00,2026-06-15 22:30:18,-477.700,NaT,NaT,NaN
4,2026-05-03,RE,DB Regio,RE3,Schaffhausen,2026-05-03 14:00:00,2026-05-03 12:00:42,-119.300,2026-05-03 13:58:00,2026-05-03 11:59:54,REAL
5,2026-04-07,EC,SBB,EC,Schaffhausen,2026-04-07 11:34:00,2026-04-07 10:41:01,-53.000,2026-04-07 11:31:00,2026-04-07 10:35:59,REAL
6,2025-09-07,IR,SBB,IR15,Palézieux,2025-09-07 21:57:00,2025-09-07 21:11:38,-45.400,2025-09-07 21:56:00,NaT,PROGNOSE
7,2025-09-03,IR,RhB,IR38,Chur,2025-09-03 12:58:00,2025-09-03 12:14:51,-43.200,NaT,NaT,NaN
8,2025-08-13,IR,SBB,IR13,Wil SG,2025-08-13 12:56:00,2025-08-13 12:16:42,-39.300,NaT,NaT,NaN
9,2025-08-07,IR,SBB,IR90,Brig,2025-08-07 10:26:00,2025-08-07 09:48:06,-37.900,NaT,NaT,NaN


In [25]:
# 6.5 Size of the extreme values: how many rows, how far off, on which days
q(f"""
SELECT CASE WHEN dep_delay_min < -1440 THEN 'a: < -1 day'
            WHEN dep_delay_min < -60   THEN 'b: -1 day .. -1 h'
            WHEN dep_delay_min < -5    THEN 'c: -1 h .. -5 min'
            WHEN dep_delay_min > 180   THEN 'd: > 3 h'
       END AS bucket,
       count(*) AS rows, count(DISTINCT operating_day) AS days,
       min(dep_delay_min) AS min_delay, max(dep_delay_min) AS max_delay,
       mode(operator_abbr) AS main_operator
FROM trains
WHERE {SCOPE} AND dep_status = 'REAL' AND (dep_delay_min < -5 OR dep_delay_min > 180)
GROUP BY ALL ORDER BY bucket
""")

,bucket,rows,days,min_delay,max_delay,main_operator
0,a: < -1 day,1,1,"-66,110,897.000","-66,110,897.000",RBS
1,b: -1 day .. -1 h,4,4,"-1,437.317",-119.300,SBB
2,c: -1 h .. -5 min,146,107,-52.983,-5.017,SBB
3,d: > 3 h,31,16,183.067,320.683,SBB


> **Finding 6.4–6.5.** The 151 departures more than 5 min early are **not real early departures** but data errors of a few kinds:
> - **Placeholder date:** one RBS RE5 departure at Lohn-Lüterkofen has `dep_actual = 1899-12-30 00:00` (the "zero date" of spreadsheet software), hence −66 million min. It alone explains the RE mean.
> - **Planned date one day off:** two EC departures at Basel SBB have a planned time stamped on the next day (e.g. planned 9 Jan 07:38, measured 8 Jan 07:40): −1,437 and −1,343 min.
> - **Time-zone offset:** a DB Regio RE3 at Schaffhausen is measured exactly ≈ 2 h early at arrival and departure, i.e. UTC instead of Swiss summer time (DB Regio is excluded anyway).
> - **Early "departure" at the origin station:** the bulk (146 rows over 107 days, mostly SBB): IR / IC trains at their first stop (Brig, Chur, Wil…) "leaving" 25–45 min early, with no arrival. Most likely the train set moving into the platform, recorded as a departure.
>
> On the late side, the 31 departures over 3 h (16 days, max 5 h 20) are plausible major disruptions and are kept.

In [26]:
# 6.6 Departure before arrival at the same stop: by how much? (REAL both, minutes)
q(f"""
WITH x AS (
  SELECT date_diff('second', arr_actual, dep_actual) / 60.0 AS dwell_min, is_swiss
  FROM trains
  WHERE {SCOPE} AND arr_status = 'REAL' AND dep_status = 'REAL' AND dep_actual < arr_actual
)
SELECT is_swiss, count(*) AS rows,
       quantile_cont(dwell_min, [0.01, 0.1, 0.5, 0.9]) AS dwell_p01_p10_p50_p90,
       avg((dwell_min > -1)::INT) AS share_within_1min
FROM x GROUP BY ALL
""")

,is_swiss,rows,dwell_p01_p10_p50_p90,share_within_1min
0,False,7,"[-6.819999999999999, -5.199999999999999, -2.0,...",0.000
1,True,39194,"[-0.9666666666666667, -0.7166666666666667, -0....",0.991


> **Finding 6.6.** Departure-before-arrival is **measurement noise**: at Swiss stops (39,194 rows), 99.1 % are within 1 min (arrival and departure are detected at different points of the platform). The 7 foreign rows are irrelevant (foreign stops are dropped).
>
> **Decisions for `journeys.py`:**
> 1. A REAL time giving a delay outside **[−5 min, +6 h]** is invalid and treated as missing (151 departures, 0.002 %; no late value is affected).
> 2. **Pass-through rows** are not prediction points or target stops, but stay in the data as network observations.
> 3. A **negative dwell** (departure measured before arrival) is kept as noise.

## 7. Journey reconstruction

One journey = one `(operating_day, trip_id)`. Stops are ordered by planned time; the checks below tell us whether that ordering is reliable and whether `trip_key` really identifies a single train run.

In [27]:
# 7.1 Build the temporary table `journeys`: in-scope rows ordered by planned time within each run
con.execute(f"""
CREATE OR REPLACE TEMP TABLE journeys AS
SELECT *,
  row_number() OVER w AS stop_seq,
  count(*) OVER (PARTITION BY trip_key) AS n_stops,
  lead(arr_planned) OVER w AS next_arr_planned,
  lead(station_id)  OVER w AS next_station_id
FROM trains
WHERE {SCOPE}
WINDOW w AS (PARTITION BY trip_key
             ORDER BY coalesce(arr_planned, dep_planned), coalesce(dep_planned, arr_planned), station_id)
""")
q("SELECT count(*) AS rows, count(DISTINCT trip_key) AS journeys FROM journeys")

,rows,journeys
0,10290359,1157584


In [28]:
# 7.2 Is one (operating_day, trip_id) exactly one train run? Counts of journeys failing each check
q("""
WITH j AS (
  SELECT trip_key,
         count(*) AS n_stops,
         count(DISTINCT station_id) AS n_stations,
         count(DISTINCT operator_id) AS n_operators,
         count(DISTINCT category) AS n_categories,
         count(DISTINCT train_number) AS n_train_numbers,
         bool_or(stop_seq = 1 AND arr_planned IS NOT NULL) AS first_has_arr,
         bool_or(stop_seq = n_stops AND dep_planned IS NOT NULL) AS last_has_dep,
         count(*) FILTER (WHERE dep_planned > next_arr_planned) AS order_violations,
         count(*) FILTER (WHERE station_id = next_station_id) AS consecutive_duplicates
  FROM journeys GROUP BY trip_key
)
SELECT
  count(*)                                   AS journeys,
  count(*) FILTER (WHERE n_stops = 1)        AS single_stop,
  count(*) FILTER (WHERE n_stations < n_stops) AS repeated_station,
  count(*) FILTER (WHERE n_operators > 1)    AS multi_operator,
  count(*) FILTER (WHERE n_categories > 1)   AS multi_category,
  count(*) FILTER (WHERE n_train_numbers > 1) AS multi_train_number,
  count(*) FILTER (WHERE first_has_arr)      AS first_stop_has_arr,
  count(*) FILTER (WHERE last_has_dep)       AS last_stop_has_dep,
  count(*) FILTER (WHERE order_violations > 0) AS order_violation,
  count(*) FILTER (WHERE consecutive_duplicates > 0) AS consecutive_duplicate
FROM j
""").T

,0
journeys,1157584
single_stop,18
repeated_station,194
multi_operator,0
multi_category,0
multi_train_number,0
first_stop_has_arr,43
last_stop_has_dep,158
order_violation,409
consecutive_duplicate,191


> **Finding 7.1–7.2.** 10.29 M in-scope rows form **1,157,584 journeys**, and a `(operating_day, trip_id)` is one train run: no journey has several operators, categories or train numbers. Exceptions are very rare (≈ 0.09 % of journeys): 18 single-stop journeys, 194 with a station visited twice (191 of them consecutively, i.e. a duplicated row), 43 whose first stop has an arrival, 158 whose last stop has a departure, and 409 with a planned-order violation.
>
> **Decision:** flag these journeys (`is_valid_journey = false`, together with the planned inconsistencies of 7.8) and exclude them from prediction points and targets.


In [29]:
# 7.3 Chart: number of stops per journey (40 = 40 or more)
stops = q("""
SELECT category, least(n_stops, 40) AS n_stops, count(DISTINCT trip_key) AS journeys
FROM journeys GROUP BY ALL ORDER BY n_stops
""")
fig = px.bar(stops, x="n_stops", y="journeys", color="category", barmode="stack",
             color_discrete_map=CAT_COLORS, category_orders={"category": list(SCOPE_CATEGORIES)},
             title="Stops per journey (40 = 40 or more)")
fig.update_layout(xaxis_title="stops", yaxis_title="journeys", legend_title=None)
fig.show()

In [30]:
# 7.4 Share of journeys running past midnight (planned times on the next calendar day)
q("""
SELECT avg((max_t::DATE > operating_day)::INT) AS share_past_midnight
FROM (SELECT trip_key, any_value(operating_day) AS operating_day,
             max(coalesce(dep_planned, arr_planned)) AS max_t
      FROM journeys GROUP BY trip_key)
""")

,share_past_midnight
0,0.042


> **Finding 7.3–7.4.**
- Typical lengths: IC 4–15 stops, IR 3–22, RE 3–23, EC 8–10 (these counts still include foreign stops).
- 18 RE journeys have a single stop (junk) and a handful of IR runs have 29–40 stops (to look at).
- **4.2 % of journeys run past midnight**, so every time computation must use the full timestamps, never `operating_day` + clock time.

In [31]:
# 7.5 Chart: mean REAL departure delay by stop number (valid delays only)
along = q("""
SELECT category, stop_seq, avg(dep_delay_min) AS mean_delay, count(*) AS n
FROM journeys
WHERE is_swiss AND dep_status = 'REAL' AND stop_seq <= 25
  AND dep_delay_min BETWEEN -5 AND 360   -- validity rule from 6.6
GROUP BY ALL ORDER BY stop_seq
""")
fig = px.line(along, x="stop_seq", y="mean_delay", color="category", markers=True,
              color_discrete_map=CAT_COLORS, category_orders={"category": list(SCOPE_CATEGORIES)},
              title="Mean departure delay along the journey", hover_data={"n": ":,"})
fig.update_layout(xaxis_title="stop number", yaxis_title="mean delay (min)", legend_title=None,
                  hovermode="x unified")
fig.show()

> **Finding 7.5.**
- On average **delay hardly builds up along a journey**: IC, IR and RE stay around 1.0–1.6 min at every stop number, so timetable slack absorbs most of what is gained.
- EC is the exception: from ≈ 2 min at the first stops to 12–13 min around stops 8–9, where many ECs enter Switzerland with delay from Italy or Germany. (The first run showed −177 min at RE stop 4: the 1899 row from 6.4, now excluded by the validity rule.)
- Implication for modelling: on average persistence will be hard to beat; the signal is in the tail, the trains that do gain or lose delay.

In [32]:
# 7.6 One example journey: a long domestic IC run on a weekday
example = q("""
SELECT trip_key FROM journeys
WHERE category = 'IC' AND n_stops >= 10 AND operating_day = DATE '2026-03-10'
GROUP BY trip_key HAVING bool_and(is_swiss)   -- domestic run only
ORDER BY trip_key LIMIT 1
""")["trip_key"][0]
trip = q(f"""
SELECT stop_seq, station_name, line_name, arr_planned, arr_actual, arr_status, arr_delay_min,
       dep_planned, dep_actual, dep_status, dep_delay_min, is_cancelled
FROM journeys WHERE trip_key = '{example}' ORDER BY stop_seq
""")
print(example)
trip

2026-03-10|ch:1:sjyid:100001:1609-001


,stop_seq,station_name,line_name,arr_planned,arr_actual,arr_status,arr_delay_min,dep_planned,dep_actual,dep_status,dep_delay_min,is_cancelled
0,1,Lausanne,IC51,NaT,NaT,NaN,NaN,2026-03-10 06:04:00,2026-03-10 06:05:28,REAL,1.467,False
1,2,Renens VD,IC51,2026-03-10 06:10:00,2026-03-10 06:10:14,REAL,0.233,2026-03-10 06:11:00,2026-03-10 06:12:08,REAL,1.133,False
2,3,Yverdon-les-Bains,IC51,2026-03-10 06:30:00,2026-03-10 06:29:26,REAL,-0.567,2026-03-10 06:31:00,2026-03-10 06:31:56,REAL,0.933,False
3,4,Neuchâtel,IC51,2026-03-10 06:51:00,2026-03-10 06:49:53,REAL,-1.117,2026-03-10 06:53:00,2026-03-10 06:53:31,REAL,0.517,False
4,5,Biel/Bienne,IC51,2026-03-10 07:12:00,2026-03-10 07:11:25,REAL,-0.583,2026-03-10 07:19:00,2026-03-10 07:19:34,REAL,0.567,False
5,6,Grenchen Nord,IC51,2026-03-10 07:28:00,2026-03-10 07:28:26,REAL,0.433,2026-03-10 07:29:00,2026-03-10 07:29:41,REAL,0.683,False
6,7,Moutier,IC51,2026-03-10 07:37:00,2026-03-10 07:36:45,REAL,-0.250,2026-03-10 07:38:00,2026-03-10 07:39:58,REAL,1.967,False
7,8,Delémont,IC51,2026-03-10 07:48:00,2026-03-10 07:48:10,REAL,0.167,2026-03-10 07:52:00,2026-03-10 07:52:35,REAL,0.583,False
8,9,Laufen,IC51,2026-03-10 08:05:00,2026-03-10 08:03:52,REAL,-1.133,2026-03-10 08:06:00,2026-03-10 08:07:31,REAL,1.517,False
9,10,Basel SBB,IC51,2026-03-10 08:26:00,2026-03-10 08:25:04,REAL,-0.933,NaT,NaT,NaN,NaN,False


In [33]:
# 7.7 Chart: delay along the example journey
pts = pd.concat([
    trip[["station_name", "arr_planned", "arr_delay_min", "arr_status"]]
        .rename(columns={"arr_planned": "planned", "arr_delay_min": "delay", "arr_status": "status"})
        .assign(event="arrival"),
    trip[["station_name", "dep_planned", "dep_delay_min", "dep_status"]]
        .rename(columns={"dep_planned": "planned", "dep_delay_min": "delay", "dep_status": "status"})
        .assign(event="departure"),
]).dropna(subset=["planned"]).sort_values("planned")
fig = px.line(pts, x="planned", y="delay", markers=True, hover_data=["station_name", "event", "status"],
              title=f"Delay along one IC journey ({example})")
fig.update_traces(line_color=SERIES[0])
fig.update_layout(xaxis_title="planned time", yaxis_title="delay (min)")
fig.show()

In [34]:
# 7.8 Ordering problems per day: planned order, planned dwell, and measured times going backwards
order_daily = q("""
WITH x AS (
  SELECT operating_day, dep_planned, arr_planned, next_arr_planned, dep_status, dep_actual,
         lead(arr_actual) OVER (PARTITION BY trip_key ORDER BY stop_seq) AS next_arr_actual,
         lead(arr_status) OVER (PARTITION BY trip_key ORDER BY stop_seq) AS next_arr_status
  FROM journeys
)
SELECT operating_day,
       count(*) FILTER (WHERE dep_planned > next_arr_planned)  AS planned_order_violations,
       count(*) FILTER (WHERE arr_planned > dep_planned)       AS planned_dep_before_arr,
       count(*) FILTER (WHERE dep_status = 'REAL' AND next_arr_status = 'REAL'
                          AND next_arr_actual < dep_actual)     AS actual_next_arr_before_dep
FROM x GROUP BY operating_day ORDER BY operating_day
""")
order_daily["total"] = order_daily.iloc[:, 1:].sum(axis=1)
print(order_daily.iloc[:, 1:].sum())
dst = pd.to_datetime(["2025-10-26", "2026-03-29"])  # daylight-saving changes
display(order_daily[pd.to_datetime(order_daily["operating_day"]).isin(dst)])
order_daily.nlargest(10, "total")

planned_order_violations        476
planned_dep_before_arr         3961
actual_next_arr_before_dep     5638
total                         10075
dtype: int64


,operating_day,planned_order_violations,planned_dep_before_arr,actual_next_arr_before_dep,total
86,2025-10-26,42,48,11,101
240,2026-03-29,0,5,12,17


,operating_day,planned_order_violations,planned_dep_before_arr,actual_next_arr_before_dep,total
14,2025-08-15,44,20,76,140
274,2026-05-02,15,108,12,135
275,2026-05-03,15,106,12,133
99,2025-11-08,0,110,11,121
100,2025-11-09,0,105,13,118
211,2026-02-28,27,65,12,104
212,2026-03-01,28,64,12,104
85,2025-10-25,42,48,11,101
86,2025-10-26,42,48,11,101
281,2026-05-09,0,78,12,90


> **Finding 7.8.** Ordering problems are rare: over 10.29 M rows, 476 planned departures later than the next planned arrival, 3,961 planned departures before the planned arrival at the same stop, and 5,638 measured arrivals at the next stop before the measured departure (≈ 0.1 % of rows in total).
> - **No daylight-saving artefact:** 26 Oct 2025 has exactly the same counts as the day before, Saturday 25 Oct (42 / 48 / 11), and 29 Mar 2026 is normal (17). The 26 Oct peak in section 5 is a weekend of works, not a clock problem.
> - **Planned inconsistencies come in weekend pairs** with identical counts (25–26 Oct, 8–9 Nov 2025, 28 Feb–1 Mar, 2–3 May 2026…): they belong to the special timetables published for engineering works, where some stop times are inconsistent.
> - **Measured next-arrival-before-departure** is a steady background of ≈ 11–13 rows per day, plus a few peaks (76 on 15 Aug 2025): measurement noise on very short segments.
>
> **Decisions for `journeys.py`:** order stops by planned time (as here); flag journeys with a planned-order violation or a negative planned dwell (`has_plan_inconsistency`) and exclude them from prediction points and targets; keep negative measured segment times as noise (clip measured running times at 0 when used as features).


## 8. Prediction points preview

Prediction points are measured (`REAL`) departures of in-scope trains. This is the upper bound on the training set before horizon labels are built (Day 2).

In [35]:
# 8.1 Departures and prediction points (REAL, not cancelled) per month, Swiss stops
pp = q(f"""
SELECT strftime(operating_day, '%Y-%m') AS month,
       count(*) FILTER (WHERE dep_planned IS NOT NULL)                  AS departures,
       count(*) FILTER (WHERE dep_status = 'REAL' AND NOT is_cancelled) AS prediction_points
FROM trains WHERE {SCOPE} AND is_swiss
GROUP BY ALL ORDER BY month
""")
pp["share"] = pp["prediction_points"] / pp["departures"]
pp

,month,departures,prediction_points,share
0,2025-08,583552,549857,0.942
1,2025-09,561300,537900,0.958
2,2025-10,571391,551035,0.964
3,2025-11,556244,533901,0.960
4,2025-12,563528,550953,0.978
5,2026-01,557601,543578,0.975
6,2026-02,506321,486082,0.960
7,2026-03,557704,536849,0.963
8,2026-04,542564,520964,0.960
9,2026-05,566205,537204,0.949


> **Finding 8.1.**
- Over 426 days, **7.45 M prediction points** out of 7.78 M in-scope departures at Swiss stops (95.7 %), i.e. 514–551 k per month, ≈ 17–18 k per day.
- The share is highest in December–January (97.5–97.8 %) and lowest in summer (93.8–94.6 % in Aug 2025 and Jul–Aug 2026), following the construction season seen in section 5.
- By split: train (Aug 2025 – Apr 2026) 4.81 M, validation (May) 0.54 M, test (June) 0.52 M, simulated production (Jul–Sep) 1.58 M.

- These are **upper bounds**: the exclusions decided above (FART / DB / DB Regio, stations < 80 % REAL, invalid journeys, pass-through rows, delays outside [−5 min, +6 h]) and the horizon labels of Day 2 will reduce them.
- Volume is not a constraint; training can subsample prediction points if needed, as the roadmap foresees.


## 9. Findings and decisions

### Key numbers

- **Dataset:** 426 days (1 Aug 2025 – 30 Sep 2026), 76.0 M train rows, no missing day.
- **Scope:** IC / IR / RE / EC, 10.3 M rows and **1.16 M train runs** (≈ 2,700 per day): RE 561 k, IR 387 k, IC 179 k, EC 30 k.
- **Measurement:** **95.7 % `REAL`** at Swiss stops vs 0.7 % abroad. No feed outage (daily share ≥ 85 %).
- **Delays:** median departure delay < 1 min for IC / IR / RE, 1.5 min for EC. More than 3 min late: IC 10 %, IR 8 %, RE 9 %, EC 29 %. On average delay hardly builds up along a journey (except EC entering Switzerland).
- **Cancellations:** 1.0 % of runs fully cancelled, 6.1 % partially. Both double at weekends and peak in summer: planned engineering works, which also explain every low-`REAL` day.
- **Journeys:** one `(operating_day, trip_id)` = one train run; 0.09 % of journeys are anomalous; 4.2 % run past midnight.
- **Prediction points:** 7.45 M upper bound; **7.38 M after all rules** (`journeys.py`): train 4.78 M, validation 0.53 M, test 0.51 M, simulated production 1.56 M. 1,109,061 journeys, of which 619 invalid.

### Decisions, implemented in `src/swissdelay/data/journeys.py`

| # | Topic | Decision | Evidence |
|---|---|---|---|
| 1 | Scope | `category` in IC, IR, RE, EC. The line-name prefix adds nothing (≈ 190 hidden runs, 0.02 %). International long-distance trains (ICE, TGV, RJX, RJ, NJ, EN) are network context only | 2.1–2.3 |
| 2 | Foreign stops | Keep Swiss-numbered stops only. Cross-border runs keep their Swiss section, flagged `enters_from_abroad`; foreign-only runs (≈ 49 k) are dropped. Foreign stops are *not* used as history: they have no measured times | 3.1–3.4 |
| 3 | Observed times | Keep every row (planned times define order, targets and slack), but only `REAL` times count as observed; other actual times are treated as missing | 4.1–4.2 |
| 4 | Operators | FART, DB and DB Regio are excluded from prediction points and targets (0.8 % of Swiss departures), kept as context | 4.3 |
| 5 | Stations | A station is a valid prediction / target station if ≥ 80 % of its in-scope departures are `REAL` over the training months (Aug 2025 – Apr 2026). This also catches Swiss-numbered stations in Italy and Basel Bad Bf | 4.4–4.6 |
| 6 | Invalid delays | A `REAL` time giving a delay outside [−5 min, +6 h] is treated as missing (151 departures) | 6.1–6.5 |
| 7 | Pass-through rows | Not prediction points or targets; kept as network observations | 6.3 |
| 8 | Noise | Negative measured dwell and negative measured running times are kept as noise (running times clipped at 0 in features) | 6.6, 7.8 |
| 9 | Stop order | Stops ordered by planned time; all time computations use full timestamps | 7.1, 7.4 |
| 10 | Invalid journeys | `is_valid_journey = false` for repeated stations, a first stop with an arrival, a last stop with a departure and planned-order violations (619 journeys, 0.06 %); excluded from prediction points and targets. A stop with a negative planned dwell (works timetables) is flagged `is_plan_inconsistent` and only that stop is excluded. Journeys with a single Swiss stop (≈ 24 k, mostly cross-border) are valid but have nothing to predict | 7.2, 7.8 |
| 11 | Cancelled stops | Not prediction points or targets | 5, 8 |

### Open points for Day 2

- **Disruption days:** worst 5 % of days (≈ 21) by a combination of mean delay and cancellations.
- **Validation and test are more disrupted** than training (full cancellations 1.3–1.6 % per month from May 2026 vs 0.6–0.9 % before): report each split's baseline level next to model errors.
- **Persistence will be strong** on average (Finding 7.5): the evaluation must look at the tails (delay severity, disruption days).
- **Line variable:** use the full `line_name` (e.g. `IRVAE`, `IC5`), not its prefix.
- **IR runs with 29–40 stops:** check whether they are real.